# Diabetes Risk Prediction — and Shipping It
### Week 4, Session 2 — the project

This morning you learned how trees, forests and boosting work. This afternoon you build
a real classifier on 253,680 real survey responses, tune it properly, track the
experiments, and then get it out of the notebook and into a running application
alongside last week's insurance-charges regressor.

**The headline lesson of the whole day:** deployment is not something that happens to a
good model at the end. It is a repeatable process — train, log, save, serve, containerise
— and the model is just an artifact that flows through it. Do it once for a classifier
and once for a regressor and you will see that the process does not care which is which.

**Session plan**

| Phase | What we do | Time |
|---|---|---|
| 1 | Frame the problem, meet the data | 20 min |
| 2 | Baseline, and choosing a metric that isn't a lie | 20 min |
| 3 | Model bake-off: KNN, tree, forest, boosting | 20 min |
| 4 | Hyperparameter tuning with GridSearchCV and friends | 30 min |
| 5 | The decision that actually matters: the threshold | 15 min |
| 6 | MLflow — tracking both models | 15 min |

After that, the deployment half of the session runs from `app/README.md`.

> **This is not a medical device.** It is a screening-style risk score learned from a
> phone survey. Nothing here diagnoses anybody, and we will come back to why at the end.

---
## Setup

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import (train_test_split, cross_validate, StratifiedKFold,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve,
                             ConfusionMatrixDisplay)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')

ARTIFACTS = Path('artifacts')
ARTIFACTS.mkdir(exist_ok=True)
print('Ready. Artifacts will be written to', ARTIFACTS.resolve())

---
# Phase 1 — Frame the problem, meet the data (20 min)

## What are we actually predicting?

**CDC Diabetes Health Indicators**, built from the 2015 Behavioral Risk Factor
Surveillance System (BRFSS) — an annual CDC telephone survey. 253,680 responses,
21 features, one binary target.

`Diabetes_binary` is 1 if the respondent reported **diabetes or prediabetes**, 0
otherwise.

Three things about this framing that matter more than any modelling choice:

1. **Everything is self-reported.** Nobody was weighed or blood-tested. `BMI` is derived
   from what people said their height and weight were, and people are optimistic.
2. **It is cross-sectional, not longitudinal.** We are not predicting who *will get*
   diabetes. We are predicting who *already has it and knows it*. A useful screening
   proxy; not a prognosis.
3. **Undiagnosed diabetics are labelled 0.** They are in the negative class. Any model
   trained here is partly learning "who has been to a doctor recently".

> **Instructor note.** Ten minutes on this section is not wasted time. Every one of those
> three points is a limitation you can only find by reading about how the data was
> collected — none of them is visible in `df.describe()`. This is the difference between
> someone who runs models and someone who can be trusted with one.

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/891/data.csv'

def load_diabetes_data():
    '''UCI first, local file as fallback.'''
    try:
        df = pd.read_csv(DATA_URL)
        print(f'Loaded {len(df):,} rows from UCI.')
    except Exception as e:
        print(f'UCI fetch failed ({e}). Falling back to local file.')
        df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')
        print(f'Loaded {len(df):,} rows from local CSV.')
    return df.drop(columns=['ID'], errors='ignore')

df = load_diabetes_data()
print('Shape:', df.shape)
df.head()

## The codebook

Survey data is encoded, and an encoded column you have not looked up is a bug waiting to
happen. `Age = 9` is not nine years old.

In [ ]:
codebook = pd.DataFrame([
    ('HighBP',               '0/1',        'Told by a doctor they have high blood pressure'),
    ('HighChol',             '0/1',        'Told they have high cholesterol'),
    ('CholCheck',            '0/1',        'Cholesterol checked in the last 5 years'),
    ('BMI',                  '12-98',      'Body Mass Index (self-reported height/weight)'),
    ('Smoker',               '0/1',        'Smoked 100+ cigarettes in their life'),
    ('Stroke',               '0/1',        'Ever told they had a stroke'),
    ('HeartDiseaseorAttack', '0/1',        'Coronary heart disease or myocardial infarction'),
    ('PhysActivity',         '0/1',        'Physical activity in the past 30 days (not job)'),
    ('Fruits',               '0/1',        'Eats fruit 1+ times per day'),
    ('Veggies',              '0/1',        'Eats vegetables 1+ times per day'),
    ('HvyAlcoholConsump',    '0/1',        'Heavy drinker (14+/wk men, 7+/wk women)'),
    ('AnyHealthcare',        '0/1',        'Has any kind of health coverage'),
    ('NoDocbcCost',          '0/1',        'Needed a doctor in the last year but could not afford it'),
    ('GenHlth',              '1-5',        'Self-rated general health: 1 excellent to 5 poor'),
    ('MentHlth',             '0-30',       'Days of poor mental health in the last 30'),
    ('PhysHlth',             '0-30',       'Days of poor physical health in the last 30'),
    ('DiffWalk',             '0/1',        'Serious difficulty walking or climbing stairs'),
    ('Sex',                  '0/1',        '0 = female, 1 = male'),
    ('Age',                  '1-13',       'Age band: 1 = 18-24, then 5-year bands, 13 = 80+'),
    ('Education',            '1-6',        '1 = never attended to 6 = college graduate'),
    ('Income',               '1-8',        '1 = under $10k to 8 = $75k or more'),
], columns=['column', 'range', 'meaning'])

pd.set_option('display.max_colwidth', 70)
codebook

Note `Age`, `GenHlth`, `Education` and `Income` are **ordinal** — the numbers are ordered
bands, not quantities. A tree handles that natively (it only asks "greater than?"). A
linear model treats them as evenly spaced, which is an approximation we are choosing to
accept.

## Health check on the data

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print('Duplicate rows:', f'{df.duplicated().sum():,}')
print()
print('Target balance:')
counts = df['Diabetes_binary'].value_counts().rename({0: 'no diabetes', 1: 'diabetes'})
print(counts.to_string())
print(f'\nPositive rate: {df["Diabetes_binary"].mean():.2%}')
print(f'Imbalance ratio: {counts.iloc[0] / counts.iloc[1]:.1f} to 1')

> **Instructor note — the duplicates question, and it is a genuinely different answer
> from Week 3.** There are about 24,000 duplicate rows here. In the insurance dataset we
> dropped them, because identical continuous charges to the cent almost certainly meant
> a re-upload artifact.
>
> Here, every column is a coarse survey answer: 0/1 flags and small integer bands. Out of
> 253,680 people, thousands of them genuinely gave identical answers to 21 coarse
> questions. These are real, distinct respondents. Dropping them would delete real
> people and bias the sample towards the unusual.
>
> **We keep them.** Ask the room to argue the other side before you say so — the point is
> that "always drop duplicates" is not a rule, it is a judgement that depends on what a
> row means.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

df['Diabetes_binary'].value_counts().plot.bar(ax=axes[0], color=['#4c72b0', '#c44e52'])
axes[0].set_title(f'Target: {df["Diabetes_binary"].mean():.1%} positive')
axes[0].set_xticklabels(['no diabetes', 'diabetes'], rotation=0)

sns.histplot(data=df, x='BMI', hue='Diabetes_binary', bins=50, stat='density',
             common_norm=False, ax=axes[1])
axes[1].set_xlim(12, 60); axes[1].set_title('BMI by outcome')

rate_by_age = df.groupby('Age')['Diabetes_binary'].mean()
rate_by_age.plot(ax=axes[2], marker='o')
axes[2].set_title('Diabetes rate by age band'); axes[2].set_ylabel('rate')

plt.tight_layout(); plt.show()

In [ ]:
# Which features shift the rate most? A quick, honest first look.
rates = []
for col in ['HighBP', 'HighChol', 'DiffWalk', 'HeartDiseaseorAttack', 'Stroke',
            'PhysActivity', 'Smoker', 'Sex', 'Fruits', 'Veggies', 'HvyAlcoholConsump']:
    grouped = df.groupby(col)['Diabetes_binary'].mean()
    rates.append({'feature': col,
                  'rate when 0': grouped.get(0, np.nan),
                  'rate when 1': grouped.get(1, np.nan)})

rate_table = pd.DataFrame(rates)
rate_table['difference'] = (rate_table['rate when 1'] - rate_table['rate when 0']).abs()
rate_table.sort_values('difference', ascending=False).round(3).reset_index(drop=True)

`HeartDiseaseorAttack`, `DiffWalk`, `Stroke` and `HighBP` move the rate hardest — every
one of them roughly doubles or triples it. `Fruits`, `Veggies` and `Sex` barely move it
at all, which is a useful reminder that the columns people *expect* to matter and the
columns that *do* are different sets.

Keep this ordering in mind. In Phase 5 we ask the trained model which columns it actually
leans on, and the answer is **not** this list. Working out why is one of the more
interesting five minutes of the afternoon.

## The split

Stratified, so the test set has the same 13.9% positive rate as the training set. With
imbalanced data an unstratified split can hand you a test set with a noticeably different
base rate, and then your test score is measuring the split rather than the model.

In [ ]:
TARGET = 'Diabetes_binary'
FEATURES = [c for c in df.columns if c != TARGET]

X = df[FEATURES]
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print(f'Train: {X_train.shape[0]:,} rows   positive rate {y_train.mean():.3%}')
print(f'Test:  {X_test.shape[0]:,} rows   positive rate {y_test.mean():.3%}')

### A search subsample, and why it is not cheating

Cross-validating a KNN grid on 200,000 rows takes long enough that a live session stops
being a session. So we do what practitioners actually do:

- **search** hyperparameters on a 25,000-row stratified subsample of the *training* set
- **refit** the winner on the full training set
- **evaluate** once on the untouched test set

The test set is never involved in the search, so nothing leaks. The cost is that the
best hyperparameters for 25k rows may be very slightly conservative for 200k rows —
tree-based models like a little more depth when given more data. That is a known,
bounded trade and it is the right call when the alternative is not searching at all.

In [ ]:
X_search, _, y_search, _ = train_test_split(
    X_train, y_train, train_size=25_000, stratify=y_train, random_state=RANDOM_STATE)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f'Search subsample: {X_search.shape[0]:,} rows, '
      f'positive rate {y_search.mean():.3%}')

---
# Phase 2 — Baseline, and a metric that isn't a lie (20 min)

## The dumbest possible model

Before anything else: what does "predict nobody has diabetes" score?

In [ ]:
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

print(f'Accuracy of a model that always says "no diabetes": '
      f'{accuracy_score(y_test, dummy_pred):.2%}')
print(f'Recall (diabetics it finds):                        '
      f'{recall_score(y_test, dummy_pred, zero_division=0):.2%}')

**86% accuracy, and it has never once been right about a diabetic.**

Write that on the board. Any model you build today that reports 86% accuracy has, for all
you know, learned exactly this. Accuracy on imbalanced data is not a weak metric, it is an
actively misleading one.

## The metric ladder

| Metric | Question it answers | Use it when |
|---|---|---|
| Accuracy | How often am I right? | Classes are balanced and errors cost the same |
| **Precision** | When I say diabetes, how often am I right? | False alarms are expensive |
| **Recall** (sensitivity) | Of all diabetics, how many did I find? | Misses are expensive |
| F1 | Harmonic mean of the two | You need one number and both matter |
| **ROC-AUC** | How well do I *rank* people by risk? | Comparing models; threshold-independent |
| **PR-AUC** (average precision) | Ranking quality, positive class only | Imbalanced data — often the more honest one |

### Choosing, for *this* problem

A false negative is a diabetic told they are fine. A false positive is a healthy person
advised to get a cheap, non-invasive blood test.

Those costs are nowhere near equal, so **recall matters far more than precision here**,
and the ratio is roughly the ratio of harms. That is a clinical and ethical judgement, not
a statistical one — and getting a real answer means asking a clinician, not a data
scientist.

We select models on **ROC-AUC** (it grades the ranking, independent of any threshold) and
then, in Phase 5, we choose the threshold explicitly with recall in mind. Keeping those
two decisions separate is the single most useful habit in applied classification.

## A real baseline: logistic regression

In [ ]:
logreg = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
]).fit(X_train, y_train)

proba_lr = logreg.predict_proba(X_test)[:, 1]
pred_lr = (proba_lr >= 0.5).astype(int)

print(classification_report(y_test, pred_lr,
                            target_names=['no diabetes', 'diabetes'], digits=3))
print(f'ROC-AUC: {roc_auc_score(y_test, proba_lr):.4f}')
print(f'PR-AUC:  {average_precision_score(y_test, proba_lr):.4f}  '
      f'(a random ranker would score {y_test.mean():.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred_lr, display_labels=['no diabetes', 'diabetes'],
    cmap='Blues', values_format=',', ax=axes[0], colorbar=False)
axes[0].set_title('Counts')

ConfusionMatrixDisplay.from_predictions(
    y_test, pred_lr, display_labels=['no diabetes', 'diabetes'],
    cmap='Blues', normalize='true', values_format='.2%', ax=axes[1], colorbar=False)
axes[1].set_title('Normalised by true class')

plt.tight_layout(); plt.show()

**Read the bottom-left cell of the right-hand panel.** That is the proportion of actual
diabetics this model sends home reassured, and it is enormous — the model has learned
that guessing "no" is usually safe, because at a 0.5 threshold it usually is.

The ROC-AUC says the *ranking* is decent. The recall says the *decisions* are not. Both
are true at once, and Phase 5 is where we reconcile them.

> **Instructor note.** Ask the room what they would do about this before showing them.
> The two answers you will get — "rebalance the data" and "lower the threshold" — are
> both legitimate, and we do the second, because it is simpler, reversible, and does not
> distort the probabilities. `class_weight='balanced'` is the first answer and we try it
> in the grids below.

---
# Phase 3 — The bake-off (20 min)

Now the models from this morning, on this problem, scored the same way.

**Every candidate goes in a `Pipeline`.** KNN needs scaling; trees do not. Putting the
scaler inside the pipeline means it is fitted on the training folds only — do it outside
and every cross-validation score you report is contaminated by the validation fold.

In [ ]:
candidates = {
    'Dummy (always no)': DummyClassifier(strategy='most_frequent'),

    'Logistic regression': Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),

    'KNN (k=25)': Pipeline([
        ('scale', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=25))]),

    'Decision tree (depth 5)': DecisionTreeClassifier(
        max_depth=5, random_state=RANDOM_STATE),

    'Random forest': RandomForestClassifier(
        n_estimators=200, min_samples_leaf=5,
        random_state=RANDOM_STATE, n_jobs=-1),

    'Hist gradient boosting': HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=300, random_state=RANDOM_STATE),
}

rows = []
for name, model in candidates.items():
    t0 = time.time()
    scores = cross_validate(model, X_search, y_search, cv=cv,
                            scoring=['roc_auc', 'average_precision', 'recall', 'accuracy'],
                            n_jobs=-1)
    rows.append({
        'model': name,
        'ROC-AUC': scores['test_roc_auc'].mean(),
        'AUC std': scores['test_roc_auc'].std(),
        'PR-AUC': scores['test_average_precision'].mean(),
        'recall@0.5': scores['test_recall'].mean(),
        'accuracy': scores['test_accuracy'].mean(),
        'fit time (s)': scores['fit_time'].mean(),
    })
    print(f'{name:26} done in {time.time() - t0:5.1f}s')

bakeoff = pd.DataFrame(rows).set_index('model').sort_values('ROC-AUC', ascending=False)
bakeoff.round(4)

**Four things to point at in that table.**

1. The dummy's **accuracy column** is competitive with several real models. Its ROC-AUC
   is 0.5 and its recall is 0. One column is lying to you; the other two are not.
2. **KNN is the weakest real model, and look at its fit time** — near zero, the fastest
   in the table. That is not efficiency, it is procrastination: KNN has no training phase
   worth the name. It memorises the training set and does all the work at *prediction*
   time, comparing each incoming person against 20,000 stored rows. Fast to fit, slow to
   serve, and the model file is the dataset. Worst possible thing to put behind an API.
3. **The ensembles beat the single tree**, exactly as this morning predicted.
4. **`recall@0.5` is dismal across the board.** Not one of these models finds even half
   the diabetics at the default threshold. Nothing in Phase 4 will fix that, because it
   is not a hyperparameter problem. It is a threshold problem.

> **Instructor note.** If the room is quiet, ask: "which of these six would you deploy?"
> The answer they usually reach for is the top ROC-AUC row. The answer that matters is
> "none of them yet, because none of them has a threshold."

### Checkpoint

1. Why is the scaler *inside* the pipeline rather than applied to `X` beforehand?
2. The dummy scores ~86% accuracy. Give two metrics from the table that expose it, and
   say what each one is measuring.
3. KNN predicts slowly and its model file is the whole training set. What does that mean
   for a Docker container serving live requests?

---
# Phase 4 — Hyperparameter tuning (30 min)

## The idea, and the trap

Every model above ran on defaults or on numbers I picked. Tuning replaces guessing with
searching: define a set of candidate configurations, score each one by cross-validation,
keep the best.

**The trap is doing that on the test set.** If you try 200 configurations and report the
best test score, you have used the test set 200 times and your "held-out" number is now
optimistic — you have overfitted the *selection*. Everything below searches with
cross-validation on training data only. The test set stays sealed until Phase 5.

## `GridSearchCV` — every combination

Grid search is exhaustive: it tries the full Cartesian product. `n_neighbors` at 3 values
x `weights` at 2 values x `p` at 2 values = 12 fits, times 5 folds = 60 model fits.

Note the `model__` prefix. Inside a pipeline, parameters are addressed as
`<step name>__<parameter>`, so `model__n_neighbors` means "the `n_neighbors` of the step
I called `model`". Getting this wrong is everyone's first grid search error.

In [ ]:
knn_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('model', KNeighborsClassifier()),
])

knn_grid = {
    'model__n_neighbors': [15, 25, 51],
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2],                       # 1 = Manhattan, 2 = Euclidean
}

t0 = time.time()
knn_search = GridSearchCV(knn_pipe, knn_grid, scoring='roc_auc',
                          cv=cv, n_jobs=-1, return_train_score=True)
knn_search.fit(X_search, y_search)

print(f'{len(knn_search.cv_results_["params"])} configurations x {cv.n_splits} folds '
      f'in {time.time() - t0:.0f}s')
print('Best params:', knn_search.best_params_)
print(f'Best CV ROC-AUC: {knn_search.best_score_:.4f}')

`GridSearchCV` keeps a full record of the search in `cv_results_`. Read it — the winner's
score alone tells you nothing about whether the search was worth running.

In [ ]:
def search_table(search, top=10):
    '''Tidy view of a fitted search object.'''
    res = pd.DataFrame(search.cv_results_)
    cols = [c for c in res.columns if c.startswith('param_')]
    out = res[cols + ['mean_train_score', 'mean_test_score',
                      'std_test_score', 'rank_test_score']]
    out = out.rename(columns={'mean_train_score': 'train AUC',
                              'mean_test_score': 'CV AUC',
                              'std_test_score': 'CV std',
                              'rank_test_score': 'rank'})
    return out.sort_values('rank').head(top).reset_index(drop=True)

search_table(knn_search)

Three patterns to read out of that table:

- **`weights='distance'` blows the train score up to a near-perfect 1.0.** With distance
  weighting the nearest neighbour of a training point is itself, at distance zero, so it
  predicts its own label perfectly. The CV column is the one telling the truth, and it
  ranks `distance` *below* `uniform`.
- **Only `n_neighbors` really matters.** Sort by CV AUC and the table sorts itself by k:
  51 beats 25 beats 15, consistently, across every combination of the other two
  parameters. `p` and `weights` are rounding error next to it.
- **The best value is at the edge of the grid**, which is the standard signal that the
  grid was drawn too small. If 51 wins, try 101. A search that picks an endpoint has told
  you where to search next, and that is worth more than the score it reported.

## Tuning a single tree

Same mechanics, different knobs — the overfitting controls from this morning. No
pipeline this time, because a tree needs no scaling, so parameter names have no prefix.

In [ ]:
tree_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_leaf': [1, 20, 100],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced'],
}

t0 = time.time()
tree_search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    tree_grid, scoring='roc_auc', cv=cv, n_jobs=-1, return_train_score=True)
tree_search.fit(X_search, y_search)

print(f'{len(tree_search.cv_results_["params"])} configurations in {time.time() - t0:.0f}s')
print('Best params:', tree_search.best_params_)
print(f'Best CV ROC-AUC: {tree_search.best_score_:.4f}')
search_table(tree_search, top=8)

In [ ]:
# Which knob actually mattered? Average the CV score over everything else.
res = pd.DataFrame(tree_search.cv_results_)
pivot = res.pivot_table(index='param_max_depth', columns='param_min_samples_leaf',
                        values='mean_test_score', aggfunc='mean', dropna=False)

plt.figure(figsize=(7, 4.5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='viridis')
plt.title('Mean CV ROC-AUC by depth and leaf size')
plt.ylabel('max_depth'); plt.xlabel('min_samples_leaf')
plt.tight_layout(); plt.show()

for knob in ['param_criterion', 'param_class_weight']:
    print(res.groupby(knob, dropna=False)['mean_test_score'].mean().round(4).to_string())
    print()

That heatmap is the deliverable from a grid search, not the single best score. It shows
you **where the good region is and how wide it is**. A sharp peak means a fragile
setting; a broad plateau means the choice is not critical.

The two printouts underneath show what `criterion` and `class_weight` did to the average
score. Any knob that barely moves it is a knob that spent your fit budget for nothing —
next time, drop it and spend those fits on depth and leaf size instead.

> **`class_weight='balanced'` is expected to do very little to ROC-AUC.** It reweights
> the loss so the minority class counts more, which moves *where the 0.5 threshold falls*
> far more than it moves the *ranking*. ROC-AUC only grades the ranking. Watch what it
> does to recall in Phase 5 instead.

## `RandomizedSearchCV` — when the grid gets too big

The random forest grid below has 2 x 4 x 4 x 3 = 96 combinations. At 5 folds that is 480
forest fits, and forests are not cheap. `RandomizedSearchCV` samples `n_iter` of them at
random instead.

This sounds worse and usually is not. When only two or three parameters really matter,
a random sample explores *more distinct values of those* for the same budget than a grid
does, because a grid wastes its fits re-testing the same depth against parameters that do
nothing. Rule of thumb: grid when the space is small and you understand it; random when
it is large or you do not.

**This cell takes a minute or two.** Good moment to take questions.

In [ ]:
rf_grid = {
    'n_estimators': [200, 300],
    'max_depth': [8, 12, 16, None],
    'min_samples_leaf': [1, 5, 20, 50],
    'max_features': ['sqrt', 0.4, 0.7],
}

t0 = time.time()
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    rf_grid, n_iter=12, scoring='roc_auc', cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1, return_train_score=True)
rf_search.fit(X_search, y_search)

print(f'12 of {2 * 4 * 4 * 3} possible configurations, sampled at random, '
      f'in {time.time() - t0:.0f}s')
print('Best params:', rf_search.best_params_)
print(f'Best CV ROC-AUC: {rf_search.best_score_:.4f}')
search_table(rf_search, top=6)

Look at the `train AUC` column against `CV AUC`. Configurations with
`min_samples_leaf=1` sit near a perfect training score — memorising — while the ones the
search ranks highly are the restrained ones. **The gap between those two columns is the
overfitting diagnostic**, and it is free with `return_train_score=True`.

## Tuning the booster

In [ ]:
hgb_grid = {
    'learning_rate': [0.03, 0.05, 0.1],
    'max_leaf_nodes': [15, 31, 63],
    'min_samples_leaf': [20, 100],
    'max_iter': [200, 400],
    'l2_regularization': [0.0, 1.0],
}

t0 = time.time()
hgb_search = RandomizedSearchCV(
    HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    hgb_grid, n_iter=20, scoring='roc_auc', cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1, return_train_score=True)
hgb_search.fit(X_search, y_search)

print(f'Searched in {time.time() - t0:.0f}s')
print('Best params:', hgb_search.best_params_)
print(f'Best CV ROC-AUC: {hgb_search.best_score_:.4f}')
search_table(hgb_search, top=6)

## The scoreboard

In [ ]:
searches = {
    'KNN (k=25)': knn_search,
    'Decision tree (depth 5)': tree_search,
    'Random forest': rf_search,
    'Hist gradient boosting': hgb_search,
}

tuned = pd.DataFrame([
    {'model': name,
     'CV AUC (tuned)': s.best_score_,
     'best params': json.dumps({k.replace('model__', ''): v
                                for k, v in s.best_params_.items()})}
    for name, s in searches.items()
]).set_index('model')

comparison = tuned.join(bakeoff[['ROC-AUC']].rename(columns={'ROC-AUC': 'CV AUC (default)'}))
comparison['gain'] = comparison['CV AUC (tuned)'] - comparison['CV AUC (default)']
comparison = comparison.sort_values('CV AUC (tuned)', ascending=False)
comparison[['CV AUC (default)', 'CV AUC (tuned)', 'gain']].round(4)

In [ ]:
for name, params in comparison['best params'].items():
    print(f'{name}\n    {params}\n')

**Set expectations honestly here, and read the `gain` column.** For the models that
already had sensible defaults — the forest and the booster — tuning bought a few
thousandths of AUC. KNN gained several times more, because its default `k=25` was simply
a bad choice for this data, and a bad default is exactly what searching protects you
from.

So tuning is insurance, not a performance strategy. The ceiling of every one of these
models sits within a couple of points of the others, and what is left over is a fraction
of what better features or a better-chosen threshold will buy you.

Tuning is worth doing — it is cheap, automatic, and it protects you from a badly chosen
default. It is not where the wins live. Beginners spend days here and hours on the data.
Reverse that ratio.

### Checkpoint

1. Why must the search never touch `X_test`?
2. `weights='distance'` gave a train AUC of 1.0. Why is that not evidence of a good model?
3. When would you pick `RandomizedSearchCV` over `GridSearchCV`?
4. Tuning bought a few thousandths of AUC. Name two things you could do instead that
   would plausibly buy more.

---
# Phase 5 — The decision that actually matters: the threshold (15 min)

## Refit the winner on everything

The search ran on 25,000 rows. Now we take the winning configuration and fit it on the
full 202,944-row training set. More data, same hyperparameters.

In [ ]:
winner_name = comparison['CV AUC (tuned)'].idxmax()
winner_search = searches[winner_name]
final_model = winner_search.best_estimator_

print(f'Winner: {winner_name}')
print(f'Config: {winner_search.best_params_}')
print()

t0 = time.time()
final_model.fit(X_train, y_train)
print(f'Refitted on {len(X_train):,} rows in {time.time() - t0:.1f}s')

## The moment of truth — one look at the test set

In [ ]:
proba_test = final_model.predict_proba(X_test)[:, 1]

print(f'Test ROC-AUC: {roc_auc_score(y_test, proba_test):.4f}')
print(f'Test PR-AUC:  {average_precision_score(y_test, proba_test):.4f}'
      f'   (random ranker: {y_test.mean():.4f})')
print()
print('At the default 0.5 threshold:')
print(classification_report(y_test, (proba_test >= 0.5).astype(int),
                            target_names=['no diabetes', 'diabetes'], digits=3))

**Look at the recall on the `diabetes` row.** The ranking is good — AUC well above 0.8,
PR-AUC several times the random baseline — and yet at a 0.5 cut we miss most diabetics.

This is the point of the whole session. The model is fine. The **decision rule bolted on
top of it** is wrong, and it is wrong because 0.5 is a default nobody chose.

## Where 0.5 comes from, and why it is not for us

`predict()` calls positive anything with probability >= 0.5. That is the correct rule if
and only if a false positive and a false negative cost you the same amount. Ours do not.

`predict_proba()` gives the number the model actually computed. The threshold is a
separate, **business** decision — and unlike the model, you can change it in production
in one line with no retraining.

In [ ]:
def threshold_report(y_true, proba, thresholds):
    rows = []
    for t in thresholds:
        pred = (proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
        rows.append({
            'threshold': t,
            'flagged': tp + fp,
            'found (TP)': tp,
            'missed (FN)': fn,
            'false alarms (FP)': fp,
            'precision': precision_score(y_true, pred, zero_division=0),
            'recall': recall_score(y_true, pred, zero_division=0),
            'F1': f1_score(y_true, pred, zero_division=0),
        })
    return pd.DataFrame(rows)

grid_thresholds = [0.05, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60]
threshold_report(y_test, proba_test, grid_thresholds).round(3)

**Read this table across, not down.** Nothing about the model changed between rows — only
the number we compare its output against.

- At **0.5** we flag few people and miss most diabetics.
- At **0.15** we flag several times as many, find the large majority of diabetics, and
  most of those flags are false alarms.
- At **0.05** we flag well over half of everyone who walks in, to catch 95% of diabetics.
  Push this far enough and the score stops carrying information — flagging everybody is
  what the dummy model does, arrived at the long way.

There is no mathematically correct row. There is a row that matches what a false alarm
costs you versus what a miss costs you, and **someone who understands the domain has to
choose it.** Your job is to put this table in front of them.

## Choosing ours, out loud

For a screening tool where a positive means "worth a cheap blood test", a miss is much
worse than a false alarm. We target **recall of about 0.75** and accept the precision
that comes with it.

Stated as a policy: *we would rather send four people for an unnecessary test than let
one diabetic walk away believing they are fine.*

In [ ]:
TARGET_RECALL = 0.75

precisions, recalls, thresholds = precision_recall_curve(y_test, proba_test)
viable = np.where(recalls[:-1] >= TARGET_RECALL)[0]
best_idx = viable[np.argmax(precisions[:-1][viable])]   # highest precision that still hits target
CHOSEN_THRESHOLD = float(thresholds[best_idx])

print(f'Chosen threshold: {CHOSEN_THRESHOLD:.4f}')
print(f'  recall:    {recalls[best_idx]:.3f}')
print(f'  precision: {precisions[best_idx]:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fpr, tpr, _ = roc_curve(y_test, proba_test)
axes[0].plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc_score(y_test, proba_test):.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='random')
axes[0].set_xlabel('false positive rate'); axes[0].set_ylabel('true positive rate')
axes[0].set_title('ROC curve'); axes[0].legend()

axes[1].plot(recalls, precisions, lw=2,
             label=f'AP = {average_precision_score(y_test, proba_test):.3f}')
axes[1].axhline(y_test.mean(), ls='--', color='grey', label='random baseline')
axes[1].scatter([recalls[best_idx]], [precisions[best_idx]], color='crimson', zorder=5,
                s=80, label=f'chosen (thr={CHOSEN_THRESHOLD:.3f})')
axes[1].set_xlabel('recall'); axes[1].set_ylabel('precision')
axes[1].set_title('Precision-recall curve'); axes[1].legend()

sweep = np.linspace(0.02, 0.9, 120)
rep = threshold_report(y_test, proba_test, sweep)
axes[2].plot(rep['threshold'], rep['precision'], label='precision')
axes[2].plot(rep['threshold'], rep['recall'], label='recall')
axes[2].plot(rep['threshold'], rep['F1'], label='F1')
axes[2].axvline(CHOSEN_THRESHOLD, ls='--', color='crimson', label='chosen')
axes[2].axvline(0.5, ls=':', color='grey', label='default 0.5')
axes[2].set_xlabel('threshold'); axes[2].set_title('Trade-off as the threshold moves')
axes[2].legend()

plt.tight_layout(); plt.show()

In [ ]:
pred_default = (proba_test >= 0.5).astype(int)
pred_chosen  = (proba_test >= CHOSEN_THRESHOLD).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pred, title in [
        (axes[0], pred_default, 'Default threshold 0.5'),
        (axes[1], pred_chosen, f'Chosen threshold {CHOSEN_THRESHOLD:.3f}')]:
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=['no diabetes', 'diabetes'],
        cmap='Blues', values_format=',', ax=ax, colorbar=False)
    ax.set_title(f'{title}\nrecall {recall_score(y_test, pred):.3f} | '
                 f'precision {precision_score(y_test, pred):.3f}')
plt.tight_layout(); plt.show()

missed_default = ((pred_default == 0) & (y_test == 1)).sum()
missed_chosen  = ((pred_chosen == 0) & (y_test == 1)).sum()
print(f'Diabetics missed at 0.5:                 {missed_default:,}')
print(f'Diabetics missed at the chosen threshold: {missed_chosen:,}')
print(f'Difference: {missed_default - missed_chosen:,} people, from changing one number.')

> **Instructor note.** That final printed line is the punchline of the whole afternoon.
> Every hyperparameter search in Phase 4 moved AUC by a few thousandths. Changing one
> number that required no retraining at all moved thousands of real people from the
> "sent home" box to the "offered a test" box.
>
> This is also the honest answer to "which model should we deploy?" — the choice of
> algorithm mattered much less than the choice of operating point.

## What is the model leaning on?

In [ ]:
from sklearn.inspection import permutation_importance

# Permutation importance on a sample of the test set — the full set is slow.
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test), 20_000, replace=False)
perm = permutation_importance(
    final_model, X_test.iloc[sample_idx], y_test.iloc[sample_idx],
    n_repeats=5, scoring='roc_auc', random_state=RANDOM_STATE, n_jobs=-1)

imp = (pd.Series(perm.importances_mean, index=X_test.columns)
       .sort_values(ascending=False))

plt.figure(figsize=(9, 7))
imp.sort_values().plot.barh()
plt.title('Permutation importance (drop in ROC-AUC when the column is shuffled)')
plt.tight_layout(); plt.show()

print(imp.round(4).to_string())

Sanity-check that ranking against the codebook. If something nonsensical were at the top
we would have a leakage bug. Instead: self-rated general health, BMI, age band, blood
pressure, cholesterol. Plausible, and consistent with the published work on BRFSS.

**Now go back to the raw rate table in Phase 1 and compare.** `HeartDiseaseorAttack`,
`DiffWalk` and `Stroke` had the three biggest raw rate gaps in the whole dataset, and
here they are near the bottom, contributing almost nothing.

They have not become unimportant. Their information is already in the model, arriving
through `GenHlth` and `Age` — someone who has had a stroke and struggles to walk does not
rate their general health as excellent. Shuffle `Stroke` and the model barely notices,
because it can reconstruct most of what that column told it from columns it still has.

> **This is the correlated-feature effect, and it is the single most misread thing about
> feature importance.** "Unimportant to this model" is not "unrelated to the outcome".
> Drop `GenHlth` and retrain, and `DiffWalk` would climb straight up the table. Never let
> anyone read an importance chart as a ranking of causes.

> `Income` sits mid-table with a small but real contribution, and `Education` is lower.
> Small is not zero, and it deserves a moment of discomfort: socioeconomic status
> correlates with real diabetes risk *and* with access to the healthcare that produces a
> diagnosis in the first place. A model that flags poorer people slightly more often is
> doing something defensible *and* something worth auditing before it is used for
> anything consequential. Say that out loud to the room.

## Save the artifact

Two files. The model, and everything a consumer needs in order to use it correctly —
above all the threshold, which is not stored inside the model object.

In [ ]:
MODEL_PATH = ARTIFACTS / 'diabetes_risk_model.joblib'
META_PATH = ARTIFACTS / 'diabetes_risk_metadata.json'

joblib.dump(final_model, MODEL_PATH)

metadata = {
    'model_name': 'diabetes_risk',
    'algorithm': winner_name,
    'best_params': {k: v for k, v in winner_search.best_params_.items()},
    'features': list(X_train.columns),
    'threshold': round(CHOSEN_THRESHOLD, 4),
    'target_recall': TARGET_RECALL,
    'trained_rows': int(len(X_train)),
    'test_metrics': {
        'roc_auc': float(roc_auc_score(y_test, proba_test)),
        'pr_auc': float(average_precision_score(y_test, proba_test)),
        'recall_at_threshold': float(recall_score(y_test, pred_chosen)),
        'precision_at_threshold': float(precision_score(y_test, pred_chosen)),
        'accuracy_at_threshold': float(accuracy_score(y_test, pred_chosen)),
    },
    'data_source': DATA_URL,
    'sklearn_version': __import__('sklearn').__version__,
}
META_PATH.write_text(json.dumps(metadata, indent=2))

print(f'Saved {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024:.0f} KB)')
print(f'Saved {META_PATH}')
print()
print(json.dumps(metadata, indent=2)[:900])

In [ ]:
# Prove the round trip, exactly as the app will do it.
reloaded = joblib.load(MODEL_PATH)
meta = json.loads(META_PATH.read_text())

example = pd.DataFrame([{
    'HighBP': 1, 'HighChol': 1, 'CholCheck': 1, 'BMI': 34, 'Smoker': 0, 'Stroke': 0,
    'HeartDiseaseorAttack': 0, 'PhysActivity': 0, 'Fruits': 0, 'Veggies': 1,
    'HvyAlcoholConsump': 0, 'AnyHealthcare': 1, 'NoDocbcCost': 0, 'GenHlth': 4,
    'MentHlth': 5, 'PhysHlth': 10, 'DiffWalk': 1, 'Sex': 1, 'Age': 9,
    'Education': 4, 'Income': 3,
}])[meta['features']]

risk = reloaded.predict_proba(example)[0, 1]
print(f'Predicted risk: {risk:.1%}')
print(f'Threshold:      {meta["threshold"]:.1%}')
print(f'Decision:       {"FLAG for screening" if risk >= meta["threshold"] else "no action"}')

### Checkpoint

1. Why is the threshold stored in a metadata file rather than inside the model object?
2. Someone asks you to "make the model more accurate". Given the table in this phase,
   what do you ask them first?
3. You ship at a threshold of 0.15. Six months later the clinic complains about the
   volume of false alarms. What do you change, and what do you *not* have to change?

---
# Phase 6 — MLflow: tracking both models (15 min)

## The problem MLflow solves

You have now run about 50 model fits this afternoon. Quick — what was the CV AUC of the
random forest with `min_samples_leaf=20`? What threshold did the model you saved an hour
ago use? Which version of scikit-learn produced it?

The notebook knows, until you restart the kernel. Then it is gone, and the honest answer
becomes "I'd have to re-run it".

**MLflow is a lab notebook for model runs.** For each run it records parameters, metrics,
artifacts and the model itself, in a local folder you can browse and query. Nothing here
needs a server or an account.

In [ ]:
try:
    import mlflow
except ImportError:
    %pip install -q mlflow
    import mlflow

import mlflow.sklearn
from mlflow.models import infer_signature

# MLflow 3 deprecated the plain-folder ('./mlruns') backend. A local SQLite file is the
# supported replacement, works on every version, and is still just a file on disk.
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_registry_uri('sqlite:///mlflow.db')

def log_sklearn_model(model, name, **kwargs):
    '''One wrapper so this notebook runs on both MLflow 2.x and 3.x.

    - the destination argument was renamed `artifact_path` -> `name` in MLflow 3
    - MLflow 3 defaults to the `skops` serialiser, which refuses to save a
      ColumnTransformer; cloudpickle handles any scikit-learn object
    '''
    kwargs.setdefault('serialization_format', 'cloudpickle')
    try:
        return mlflow.sklearn.log_model(model, name=name, **kwargs)
    except TypeError:
        return mlflow.sklearn.log_model(model, artifact_path=name, **kwargs)

print('MLflow', mlflow.__version__)
print('Tracking to', mlflow.get_tracking_uri())

## Logging the search results

Every configuration the grid searches tried is already sitting in `cv_results_`. Logging
the winner of each search takes one loop.

In [ ]:
mlflow.set_experiment('week04-diabetes-classification')

for name, search in searches.items():
    with mlflow.start_run(run_name=f'search: {name}'):
        mlflow.log_params({k.replace('model__', ''): v
                           for k, v in search.best_params_.items()})
        mlflow.log_param('search_type', type(search).__name__)
        mlflow.log_param('search_rows', len(X_search))
        mlflow.log_param('algorithm', name)
        mlflow.log_metric('cv_roc_auc', search.best_score_)
        mlflow.log_metric('cv_roc_auc_std',
                          search.cv_results_['std_test_score'][search.best_index_])
        mlflow.log_metric('n_configs_tried', len(search.cv_results_['params']))
    print(f'logged {name}')

## Logging the model we are actually going to ship

This run gets the full treatment: parameters, test metrics, the chosen threshold, the
metadata file as an artifact, and the model itself with a **signature** — the recorded
schema of what goes in and what comes out, so anything loading this model later can
validate its input instead of failing mysteriously.

In [ ]:
signature = infer_signature(X_train.head(100), final_model.predict_proba(X_train.head(100)))

with mlflow.start_run(run_name='diabetes_risk — production candidate') as run:
    mlflow.log_params({k.replace('model__', ''): v
                       for k, v in winner_search.best_params_.items()})
    mlflow.log_param('algorithm', winner_name)
    mlflow.log_param('train_rows', len(X_train))
    mlflow.log_param('threshold', round(CHOSEN_THRESHOLD, 4))

    mlflow.log_metrics({
        'test_roc_auc': metadata['test_metrics']['roc_auc'],
        'test_pr_auc': metadata['test_metrics']['pr_auc'],
        'recall_at_threshold': metadata['test_metrics']['recall_at_threshold'],
        'precision_at_threshold': metadata['test_metrics']['precision_at_threshold'],
        'accuracy_at_threshold': metadata['test_metrics']['accuracy_at_threshold'],
    })

    mlflow.set_tags({'task': 'classification', 'dataset': 'CDC BRFSS 2015',
                     'week': '04', 'app_tab': 'Diabetes Risk'})

    mlflow.log_artifact(str(META_PATH))
    log_sklearn_model(final_model, 'model',
                      signature=signature, input_example=X_train.head(3))

    diabetes_run_id = run.info.run_id

print('Diabetes model logged as run', diabetes_run_id)

## The same six lines, for a completely different model

Here is the claim from the top of the notebook, tested. Last week's model predicts a
dollar amount from seven columns with a linear regression. This week's predicts a
probability from twenty-one columns with an ensemble. They have nothing in common.

**The logging code is the same.** That is what "deployment is a repeatable process, not a
property of the model" means in practice.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

def build_insurance_model():
    '''Rebuild Week 3's model from scratch — the same pipeline, in 20 lines.'''
    try:
        from huggingface_hub import list_repo_files, hf_hub_download
        repo = 'rahulvyasm/medical_insurance_data'
        csvs = [f for f in list_repo_files(repo, repo_type='dataset') if f.endswith('.csv')]
        ins = pd.read_csv(hf_hub_download(repo, csvs[0], repo_type='dataset'))
    except Exception:
        ins = pd.read_csv('medical_insurance.csv')

    ins = ins.drop_duplicates().reset_index(drop=True)
    ins['bmi_x_smoker'] = ins['bmi'] * (ins['smoker'] == 'yes').astype(int)

    numeric = ['age', 'bmi', 'children', 'bmi_x_smoker']
    categorical = ['sex', 'smoker', 'region']
    Xi = ins[numeric + categorical]
    yi = ins['charges']

    Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.2, random_state=42)
    pipe = Pipeline([
        ('prep', ColumnTransformer([('num', 'passthrough', numeric),
                                    ('cat', OneHotEncoder(drop='first'), categorical)])),
        ('model', LinearRegression()),
    ]).fit(Xi_tr, yi_tr)

    preds = pipe.predict(Xi_te)
    return pipe, Xi_tr, {'r2': r2_score(yi_te, preds),
                         'mae': mean_absolute_error(yi_te, preds)}

insurance_model, Xi_example, insurance_metrics = build_insurance_model()
print('Insurance model R2:', round(insurance_metrics['r2'], 4),
      '| MAE $', round(insurance_metrics['mae'], 2))

joblib.dump(insurance_model, ARTIFACTS / 'insurance_charges_model.joblib')
print('Saved', ARTIFACTS / 'insurance_charges_model.joblib')

In [ ]:
mlflow.set_experiment('week03-insurance-regression')

sig_reg = infer_signature(Xi_example.head(100), insurance_model.predict(Xi_example.head(100)))

with mlflow.start_run(run_name='insurance_charges — production candidate') as run:
    mlflow.log_param('algorithm', 'LinearRegression + one-hot + bmi_x_smoker interaction')
    mlflow.log_metrics({'test_r2': insurance_metrics['r2'],
                        'test_mae': insurance_metrics['mae']})
    mlflow.set_tags({'task': 'regression', 'dataset': 'medical insurance',
                     'week': '03', 'app_tab': 'Charges Predictor'})
    log_sklearn_model(insurance_model, 'model',
                      signature=sig_reg, input_example=Xi_example.head(3))
    insurance_run_id = run.info.run_id

print('Insurance model logged as run', insurance_run_id)

## Querying what you logged

`search_runs` turns the whole experiment into a DataFrame, which is how you answer "which
run was best, and what was it?" three weeks from now.

In [ ]:
runs = mlflow.search_runs(
    experiment_names=['week04-diabetes-classification'],
    order_by=['metrics.cv_roc_auc DESC'])

show_cols = [c for c in ['tags.mlflow.runName', 'params.algorithm',
                         'metrics.cv_roc_auc', 'metrics.test_roc_auc',
                         'metrics.recall_at_threshold', 'params.threshold']
             if c in runs.columns]
runs[show_cols].round(4)

## Loading a model back out

This is the piece that matters for deployment: the app does not need the notebook, the
training data, or the search. It needs a run ID.

In [ ]:
loaded = mlflow.sklearn.load_model(f'runs:/{diabetes_run_id}/model')
print('Loaded from MLflow:', type(loaded).__name__)
print(f'Prediction on our example person: {loaded.predict_proba(example)[0, 1]:.1%}')

loaded_reg = mlflow.sklearn.load_model(f'runs:/{insurance_run_id}/model')
person = pd.DataFrame([{'age': 45, 'bmi': 31.0, 'children': 2, 'bmi_x_smoker': 0.0,
                        'sex': 'male', 'smoker': 'no', 'region': 'southeast'}])
print(f'Insurance model on a 45-year-old non-smoker: ${loaded_reg.predict(person)[0]:,.0f}')

## Look at it

In a terminal, from this folder:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Then open http://127.0.0.1:5000. You get both experiments, every run, sortable metric
columns, and a diff view for comparing two runs side by side.

> **Instructor note.** Spend three minutes actually clicking around the UI with the room
> watching. The value of experiment tracking does not land from a `log_metric` call; it
> lands when someone sees twelve runs sorted by AUC and realises this is what their
> notebook comments were trying and failing to be.

### On the Model Registry

`mlflow.register_model(...)` adds names, versions and stages (Staging / Production) on
top of runs. It is the right tool when a team is promoting models on a schedule. For two
models and one Docker container it is more ceremony than we need, so we load ours from
joblib files and keep MLflow as the record of how they were made.

### Checkpoint

1. What does a model signature protect you from?
2. What could you answer three months from now with the MLflow runs that you could not
   answer from the notebook alone?
3. Your teammate has a run ID and nothing else. What can they reproduce?

---
# Phase 7 — Hand-off to the app

Two artifacts, both in `artifacts/`:

| File | What it is | Which tab |
|---|---|---|
| `diabetes_risk_model.joblib` | tuned classifier, full pipeline | Diabetes Risk |
| `diabetes_risk_metadata.json` | feature order, chosen threshold, test metrics | Diabetes Risk |
| `insurance_charges_model.joblib` | Week 3 regression pipeline | Charges Predictor |

Copy them into the app's model folder and you are done with the notebook:

```bash
mkdir -p app/models
cp artifacts/*.joblib artifacts/*.json app/models/
```

Then follow **`app/README.md`** for the deployment half of the session: one Streamlit app
with two tabs, the same two models behind a Flask JSON API, and both packaged into a
single Docker container.

The thing to notice while you do it: **nothing in the app knows or cares that one model
is a regression and the other is a classification.** Both are a `.joblib` file with a
`predict` method behind a form. That is the whole point.

---
## What this model is not

Before anyone screenshots a prediction and sends it to a friend:

1. **It is not a diagnosis.** It is a statistical association learned from a telephone
   survey, and it is wrong about a large fraction of the people it flags — look at the
   precision column again.
2. **It predicts diagnosis, not disease.** Undiagnosed diabetics are labelled 0 in the
   training data, so the model has partly learned who gets seen by a doctor.
3. **It was trained on 2015 US adults.** Applying it to a different country, a different
   decade, or a different age profile is an untested extrapolation.
4. **It uses income and education.** That is a fairness question, not just a technical
   one, and it needs answering before this is used for anything consequential.

Every deployed model needs a paragraph like this. Writing it is part of the job.

---
## Your turn

**1.** Change `TARGET_RECALL` to 0.9 and re-run Phase 5. How many more people get flagged,
and what happens to precision? Would you ship it?

**2.** Add `class_weight='balanced'` to the winning model and refit. Compare ROC-AUC and
the confusion matrix at 0.5 against the tuned threshold approach. Which gives you more
control, and why?

**3.** Drop `Income` and `Education` and retrain. How much AUC does it cost? Write two
sentences arguing for the version you would deploy.

**4.** Try `xgboost`'s `XGBClassifier` with `scale_pos_weight=6` in the Phase 4 search.
Does it beat `HistGradientBoostingClassifier` by more than one CV standard deviation?

**5.** *Deployment.* Add a third tab to the Streamlit app that shows the MLflow run table
from this notebook, so anyone using the app can see which model version they are talking
to.